# QuantBench-Inspired SOXL Intraday Data v1

This notebook is the primary evidence surface for the v1 intraday research layer. It implements a fixture smoke of the canonical minute schema, dual-source comparison, daily-safe aggregate features, leakage rules, and first signal-screen plumbing.

External references used for the design:

- QuantBench arXiv 2504.18600: unified data format, data breadth/depth, and standardized evaluation.
- Futu `request_history_kline`: paged historical K-line API, US Eastern time for US stocks, and <=60-minute coverage limits.
- Polygon/Massive aggregate bars: OHLCV custom bars, split-adjustment flag, ET aggregate windows, and missing bars when no eligible trades occur.


## Governance

- Baselines: current daily SOXL/SOXX/VIX canonical panel and no intraday feature pack.
- Direct comparison: optional external-data packs such as `tushare_us_daily_adj` and `futu_small_order_flow`, which enter daily models only through lagged cached features.
- Role: diagnose and scaffold data intake; not a promoted model/profile run.
- Target window: tiny fixture smoke in this notebook; live command path is `2024-01-01` to `2026-06-04` when secrets/OpenD are available.
- Changed layer: data and feature intake.
- Hypothesis: a QuantBench-style unified minute layer can expose provider coverage/disagreement and produce non-leaking daily aggregate features before any minute-level RL training.
- Acceptance criteria: canonical schema is stable, provider coverage is measurable, disagreement/missing bars are reported, daily features are T+1 lagged, and the first IC/backtest hooks run on a fixture.
- Expected trade-off: v1 adds data depth but no production scheduler, broker, actor, or direct minute RL changes.
- Failure mode: Futu coverage/quota or Polygon plan limits leave partial history; registry must record actual coverage instead of assuming full SOXL history.
- Comparison-table row key: `quantbench_intraday_soxl_v1`.

Closest historical success: Futu full-history daily K-line probe established Futu OHLCV as a usable comparison surface for SOXL/SOXX daily bars.

Closest historical failure/limit: Futu small-order-flow data had short coverage and remained a short-history overlay only.

Why this is not a duplicate: this v1 is minute OHLCV schema/registry/feature infrastructure across Futu and Polygon, not daily K-line replacement or small-order-flow training.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError("could not locate repo root containing src/soxl_quant")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.intraday_data import (
    CANONICAL_INTRADAY_COLUMNS,
    build_daily_intraday_features,
    canonical_intraday_path,
    intraday_feature_csv_path,
)

CARD_DIR = ROOT / "runtime/data/card/quantbench_intraday_soxl_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)
NY = "America/New_York"

print(f"repo={ROOT}")
print(f"canonical_schema={CANONICAL_INTRADAY_COLUMNS}")


repo=/Users/rziiiii/.openclaw/workspace/.worktrees/quantbench-intraday-soxl-v1
canonical_schema=['provider', 'symbol', 'interval', 'timestamp', 'date', 'session', 'open', 'high', 'low', 'close', 'volume', 'turnover', 'source_tz', 'adjusted', 'fetched_at']


In [2]:
def build_fixture_intraday() -> pd.DataFrame:
    rows = []
    days = pd.bdate_range("2024-01-02", periods=30)
    for day_idx, day in enumerate(days):
        for provider, provider_offset in (("polygon", 0.0), ("futu", 0.0004)):
            for minute in range(5):
                ts = pd.Timestamp(day.date().isoformat() + " 09:30:00", tz=NY) + pd.Timedelta(minutes=minute)
                soxx_open = 100.0 + day_idx * 0.18 + minute * 0.015
                soxx_close = soxx_open * (1.0 + 0.00035 * np.sin(day_idx / 4.0) + minute * 0.00003)
                soxl_open = 20.0 + day_idx * 0.11 + minute * 0.020
                tracking_drift = 0.0002 * np.cos(day_idx / 5.0) - provider_offset
                soxl_close = soxl_open * (1.0 + 3.0 * (soxx_close / soxx_open - 1.0) + tracking_drift)
                for symbol, open_, close_, volume_base in (
                    ("SOXL", soxl_open, soxl_close, 10000 + day_idx * 50),
                    ("SOXX", soxx_open, soxx_close, 2500 + day_idx * 20),
                ):
                    high = max(open_, close_) * (1.0 + 0.001 + minute * 0.0001)
                    low = min(open_, close_) * (1.0 - 0.001 - minute * 0.00005)
                    rows.append(
                        {
                            "provider": provider,
                            "symbol": symbol,
                            "interval": "1m",
                            "timestamp": ts,
                            "date": day.strftime("%Y-%m-%d"),
                            "session": "regular",
                            "open": open_,
                            "high": high,
                            "low": low,
                            "close": close_,
                            "volume": float(volume_base + minute * 100),
                            "turnover": 0.0,
                            "source_tz": NY,
                            "adjusted": True,
                            "fetched_at": pd.Timestamp("2026-06-05T00:00:00Z"),
                        }
                    )
    return pd.DataFrame(rows, columns=CANONICAL_INTRADAY_COLUMNS)

canonical = build_fixture_intraday()
fixture_path = CARD_DIR / "fixture_canonical_intraday.parquet"
canonical.to_parquet(fixture_path, index=False)
coverage = canonical.groupby(["provider", "symbol"]).agg(
    rows=("timestamp", "size"),
    min_timestamp=("timestamp", "min"),
    max_timestamp=("timestamp", "max"),
).reset_index()
coverage


,provider,symbol,rows,min_timestamp,max_timestamp
0,futu,SOXL,150,2024-01-02 09:30:00-05:00,2024-02-12 09:34:00-05:00
1,futu,SOXX,150,2024-01-02 09:30:00-05:00,2024-02-12 09:34:00-05:00
2,polygon,SOXL,150,2024-01-02 09:30:00-05:00,2024-02-12 09:34:00-05:00
3,polygon,SOXX,150,2024-01-02 09:30:00-05:00,2024-02-12 09:34:00-05:00


In [3]:
pivot = canonical.pivot_table(index=["symbol", "timestamp"], columns="provider", values="close")
pivot["abs_close_diff_bps"] = ((pivot["futu"] - pivot["polygon"]).abs() / pivot["polygon"] * 10000.0)
source_disagreement = {
    "mean_abs_close_diff_bps": float(pivot["abs_close_diff_bps"].mean()),
    "max_abs_close_diff_bps": float(pivot["abs_close_diff_bps"].max()),
}

expected_regular_bars = 390
bar_counts = canonical.groupby(["provider", "symbol", "date"]).size().rename("bars").reset_index()
bar_counts["missing_vs_regular_390"] = expected_regular_bars - bar_counts["bars"]
missing_summary = bar_counts.groupby(["provider", "symbol"]).agg(
    min_bars=("bars", "min"),
    max_missing=("missing_vs_regular_390", "max"),
).reset_index()

print(source_disagreement)
missing_summary


{'mean_abs_close_diff_bps': 1.9995106069314756, 'max_abs_close_diff_bps': 4.004835642684108}


,provider,symbol,min_bars,max_missing
0,futu,SOXL,5,385
1,futu,SOXX,5,385
2,polygon,SOXL,5,385
3,polygon,SOXX,5,385


In [4]:
features = build_daily_intraday_features(canonical, interval="1m", session="regular")
feature_path = CARD_DIR / "fixture_intraday_features_soxl_1m.csv"
features.to_csv(feature_path, index=False)
feature_summary = features[[
    "opening_range_30",
    "close_location_value",
    "realized_intraday_vol",
    "volume_concentration_open_30",
    "soxl_soxx_intraday_relative_volume",
    "intraday_tracking_error",
]].describe().T
feature_summary


,count,mean,std,min,25%,50%,75%,max
opening_range_30,30.0,0.007070,4.298884e-04,0.006299,0.006787,0.007094,0.007331,0.007771
close_location_value,30.0,0.772988,4.495108e-02,0.688147,0.737896,0.790251,0.808091,0.818869
realized_intraday_vol,30.0,0.002032,8.304724e-05,0.001902,0.001963,0.002029,0.002099,0.002176
volume_concentration_open_30,30.0,1.000000,7.392900e-15,1.000000,1.000000,1.000000,1.000000,1.000000
soxl_soxx_intraday_relative_volume,30.0,3.657737,6.844929e-02,3.551829,3.600487,3.653859,3.712665,3.777778
intraday_tracking_error,30.0,0.001949,2.235466e-04,0.001714,0.001770,0.001883,0.002098,0.002402


In [5]:
def time_series_ic(signal: pd.Series, returns: pd.Series, horizon: int = 1) -> dict[str, float | int]:
    forward_return = (1.0 + returns).rolling(horizon).apply(np.prod, raw=True).shift(-horizon) - 1.0
    aligned = pd.concat({"signal": signal, "forward_return": forward_return}, axis=1).dropna()
    signed = aligned["signal"] * aligned["forward_return"]
    n = int(len(aligned))
    return {
        "n_days": n,
        "ic_mean": float(signed.mean()) if n else 0.0,
        "ic_positive_ratio": float((signed > 0).mean()) if n else 0.0,
        "pearson_corr": float(aligned["signal"].corr(aligned["forward_return"])) if n > 1 else 0.0,
    }

polygon_soxl = canonical[(canonical["provider"] == "polygon") & (canonical["symbol"] == "SOXL")].sort_values("timestamp")
daily_close = polygon_soxl.groupby("date")["close"].last()
daily_returns = daily_close.pct_change().fillna(0.0)
feature_indexed = features.set_index("date")
feature_indexed.index = pd.to_datetime(feature_indexed.index)
daily_returns.index = pd.to_datetime(daily_returns.index)

signal = -feature_indexed["intraday_tracking_error"].astype(float)
signal.name = "neg_intraday_tracking_error"
ic = time_series_ic(signal, daily_returns, horizon=1)

threshold = float(feature_indexed["close_location_value"].median())
raw_position = (feature_indexed["close_location_value"] > threshold).astype(float)
position = raw_position.shift(1).fillna(0.0)
strategy_return = position * daily_returns.reindex(position.index).fillna(0.0)
equity = (1.0 + strategy_return).cumprod()
standalone = {
    "threshold": threshold,
    "total_return": float(equity.iloc[-1] - 1.0),
    "max_drawdown": float((equity / equity.cummax() - 1.0).min()),
    "num_signals": int(raw_position.sum()),
}

print({"ic": ic, "standalone": standalone})


{'ic': {'n_days': 29, 'ic_mean': -1.0046723827169262e-05, 'ic_positive_ratio': 0.0, 'pearson_corr': -0.8621584854668624}, 'standalone': {'threshold': 0.7902512947905264, 'total_return': 0.07616947832460808, 'max_drawdown': 0.0, 'num_signals': 15}}


## Leakage Rules

- Minute bars are not used directly for v1 RL training.
- `intraday_microstructure_v1` consumes only daily aggregates from `data/raw/external/intraday_features_soxl_1m.csv`.
- Every model-facing intraday feature is shifted by one trading day before entering `engineer_feature_frame()`.
- Missing pre-coverage rows are filled with zero after the lag, preserving full-horizon matrices without backfilling future minute data.
- Provider coverage and disagreement are registry/notebook evidence, not hidden failover assumptions.


In [6]:
summary = {
    "experiment": "quantbench_intraday_soxl_v1",
    "status": "fixture_smoke_only",
    "live_fetch_executed": False,
    "reason_live_fetch_not_executed": "Requires POLYGON_API_KEY and reachable Futu OpenD; this notebook records architecture smoke evidence only.",
    "canonical_schema_columns": CANONICAL_INTRADAY_COLUMNS,
    "fixture_rows": int(len(canonical)),
    "fixture_providers": sorted(canonical["provider"].unique().tolist()),
    "fixture_symbols": sorted(canonical["symbol"].unique().tolist()),
    "coverage_rows": coverage.assign(
        min_timestamp=coverage["min_timestamp"].astype(str),
        max_timestamp=coverage["max_timestamp"].astype(str),
    ).to_dict(orient="records"),
    "source_disagreement": source_disagreement,
    "missing_bar_summary": missing_summary.to_dict(orient="records"),
    "feature_rows": int(len(features)),
    "ic_screen": ic,
    "standalone_screen": standalone,
    "repo_artifacts": {
        "schema_canonical_parquet": str(canonical_intraday_path(root=ROOT).relative_to(ROOT)),
        "schema_feature_csv": str(intraday_feature_csv_path(root=ROOT).relative_to(ROOT)),
        "fixture_canonical_parquet": str(fixture_path.relative_to(ROOT)),
        "fixture_feature_csv": str(feature_path.relative_to(ROOT)),
    },
    "decision": "Do not promote; proceed to live Futu+Polygon coverage fetch when secrets/OpenD are available.",
}
summary_path = CARD_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary


{'experiment': 'quantbench_intraday_soxl_v1',
 'status': 'fixture_smoke_only',
 'live_fetch_executed': False,
 'reason_live_fetch_not_executed': 'Requires POLYGON_API_KEY and reachable Futu OpenD; this notebook records architecture smoke evidence only.',
 'canonical_schema_columns': ['provider',
  'symbol',
  'interval',
  'timestamp',
  'date',
  'session',
  'open',
  'high',
  'low',
  'close',
  'volume',
  'turnover',
  'source_tz',
  'adjusted',
  'fetched_at'],
 'fixture_rows': 600,
 'fixture_providers': ['futu', 'polygon'],
 'fixture_symbols': ['SOXL', 'SOXX'],
 'coverage_rows': [{'provider': 'futu',
   'symbol': 'SOXL',
   'rows': 150,
   'min_timestamp': '2024-01-02 09:30:00-05:00',
   'max_timestamp': '2024-02-12 09:34:00-05:00'},
  {'provider': 'futu',
   'symbol': 'SOXX',
   'rows': 150,
   'min_timestamp': '2024-01-02 09:30:00-05:00',
   'max_timestamp': '2024-02-12 09:34:00-05:00'},
  {'provider': 'polygon',
   'symbol': 'SOXL',
   'rows': 150,
   'min_timestamp': '2024-

## Conclusion

This run establishes the v1 intraday data architecture, not a production signal. The canonical schema, fixture dual-source comparison, missing-bar accounting, daily aggregate feature calculation, and IC/backtest hooks all execute in-notebook. The committed repo data artifacts are schema-only because live Futu and Polygon pulls require runtime credentials/OpenD.

Decision: keep `intraday_microstructure_v1` as a candidate data pack only. No scheduler, broker, actor, or promoted profile changes are made. Next action is to run the public fetch command with `POLYGON_API_KEY` and Futu OpenD available, then re-execute this notebook against real `SOXL/SOXX` 1-minute coverage.
